# 🌍 Tutorial 08: Generalización Out-of-Distribution (OOD)

## Del Laboratorio al Mundo Real

En este tutorial aprenderás:

- 🎯 Qué es generalización Out-of-Distribution
- ⚠️ Por qué los modelos fallan en OOD
- 🛡️ Cómo Meta-Learning ayuda con robustez OOD
- 💻 Técnicas para mejorar generalización

---

## 📖 Parte 1: El Problema OOD

### ¿Qué es Out-of-Distribution?

**OOD** ocurre cuando los datos de test son diferentes a los de training:

```
Training: 🐱 🐶 🐦 (fotos en interiores, buena iluminación)
Test:     🐱 🐶 🐦 (fotos al aire libre, diferentes ángulos, nieve)
                  ^
                  Distribution Shift!
```

### Tipos de Distribution Shift:

#### 1. **Covariate Shift** 📊
- $P(X)$ cambia, pero $P(Y|X)$ permanece igual
- Ejemplo: Modelo entrenado en imágenes día, probado en noche

#### 2. **Label Shift** 🏷️
- $P(Y)$ cambia (distribución de clases)
- Ejemplo: 50% gatos en train, 90% gatos en test

#### 3. **Concept Drift** 🌊
- $P(Y|X)$ cambia con el tiempo
- Ejemplo: Preferencias de usuarios cambian

### ¿Por qué es Crítico?

❌ **Problema Real**:
- **Robótica**: Simulación → Mundo real (sim-to-real gap)
- **Medicina**: Hospital A → Hospital B (diferentes equipos)
- **Vehículos Autónomos**: Ciudad X → Ciudad Y
- **Finanzas**: Datos históricos → Crisis económica

### El Rol de Meta-Learning:

Meta-Learning puede ayudar porque:
1. **Aprende de diversidad**: Ve muchas distribuciones durante training
2. **Adaptación rápida**: Puede ajustarse a nuevas distribuciones
3. **Regularización implícita**: Evita overfitting a una distribución


---

## 🛠️ Setup

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.append('..')

from utils.test_utils import print_success
from utils.data_utils import set_seed

set_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("✅ Setup completo!")

---

## 💻 Parte 3: Demostración - Covariate Shift con MNIST

Vamos a simular OOD aplicando transformaciones a MNIST.

In [ ]:
# Cargar MNIST
mnist_train = torchvision.datasets.MNIST(
    root='../datasets',
    train=True,
    download=True,
    transform=transforms.ToTensor()
)

mnist_test = torchvision.datasets.MNIST(
    root='../datasets',
    train=False,
    download=True,
    transform=transforms.ToTensor()
)

print(f"✅ MNIST cargado: {len(mnist_train)} train, {len(mnist_test)} test")

In [ ]:
# Crear distribuciones OOD aplicando transformaciones

def create_ood_mnist(mnist_dataset, transform_type='rotate'):
    """
    Crea versión OOD de MNIST aplicando transformaciones.
    
    Args:
        transform_type: 'rotate', 'noise', 'blur', 'invert'
    """
    if transform_type == 'rotate':
        transform = transforms.Compose([
            transforms.ToPILImage(),
            transforms.RandomRotation(degrees=45),
            transforms.ToTensor()
        ])
    elif transform_type == 'noise':
        def add_noise(img):
            return img + torch.randn_like(img) * 0.3
        transform = add_noise
    elif transform_type == 'blur':
        transform = transforms.Compose([
            transforms.ToPILImage(),
            transforms.GaussianBlur(kernel_size=5),
            transforms.ToTensor()
        ])
    elif transform_type == 'invert':
        def invert(img):
            return 1.0 - img
        transform = invert
    
    return transform


# Visualizar diferentes distribuciones
fig, axes = plt.subplots(5, 10, figsize=(15, 8))

transforms_list = ['original', 'rotate', 'noise', 'blur', 'invert']
sample_idx = 42

for row, transform_type in enumerate(transforms_list):
    for col in range(10):
        img, label = mnist_test[sample_idx + col]
        
        if transform_type != 'original':
            transform = create_ood_mnist(mnist_test, transform_type)
            img = transform(img)
        
        ax = axes[row, col]
        ax.imshow(img.squeeze(), cmap='gray')
        ax.axis('off')
        
        if col == 0:
            ax.text(-0.5, 0.5, transform_type.upper(), 
                   transform=ax.transAxes, fontsize=10, 
                   rotation=90, va='center', fontweight='bold')

plt.suptitle('Diferentes Distribuciones (OOD Shifts)', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n⚠️  Un modelo entrenado solo en 'original' puede fallar en los demás")

---

## 🛡️ Parte 4: Estrategias para Robustez OOD

### 1. **Data Augmentation Agresiva**

In [ ]:
# Augmentation agresiva durante training
robust_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.RandomRotation(30),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
    transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0)),
    transforms.ToTensor(),
    transforms.Lambda(lambda x: x + torch.randn_like(x) * 0.1),  # Ruido
])

print("✅ Augmentation agresiva expone el modelo a más variabilidad")

### 2. **Domain Randomization**

Entrenar en muchas distribuciones aleatorias:

In [ ]:
def domain_randomization(img):
    """
    Aplica transformaciones aleatorias de un conjunto diverso.
    """
    transforms_pool = [
        transforms.RandomRotation(degrees=45),
        transforms.ColorJitter(brightness=0.5, contrast=0.5),
        transforms.GaussianBlur(kernel_size=5),
        transforms.RandomPerspective(distortion_scale=0.3),
    ]
    
    # Aplicar subset aleatorio
    n_transforms = np.random.randint(1, len(transforms_pool) + 1)
    chosen = np.random.choice(len(transforms_pool), size=n_transforms, replace=False)
    
    composed = transforms.Compose([transforms_pool[i] for i in chosen])
    return composed(img)

print("✅ Domain Randomization: entrenar en muchas variaciones aleatorias")

### 3. **Meta-Learning para OOD**

Entrenar con episodios que simulan distribution shift:

In [ ]:
class MetaOODTraining:
    """
    Entrenamiento meta para robustez OOD.
    
    Idea:
    - Support set: distribución normal
    - Query set: distribución con shift
    
    El modelo aprende a adaptarse a shifts!
    """
    
    def __init__(self, base_dataset):
        self.base_dataset = base_dataset
        self.shift_types = ['rotate', 'noise', 'blur']
    
    def sample_ood_episode(self, n_way=5, k_shot=5, q_query=15):
        """
        Genera episodio con OOD shift entre support y query.
        """
        # Support: distribución original
        # Query: distribución con shift aleatorio
        
        shift_type = np.random.choice(self.shift_types)
        transform_ood = create_ood_mnist(self.base_dataset, shift_type)
        
        # ... (sample support y query con shift)
        
        return {}


print("✅ Meta-OOD Training: episodios simulan distribution shifts")
print("   → El modelo aprende a ser robusto a cambios de distribución")

---

## 📊 Parte 5: Métricas de Robustez OOD

In [ ]:
def evaluate_ood_robustness(model, dataset, ood_transforms):
    """
    Evalúa robustez del modelo bajo diferentes shifts.
    
    Args:
        model: Modelo a evaluar
        dataset: Dataset base
        ood_transforms: Lista de transformaciones OOD
    
    Returns:
        Dict[str, float]: Accuracy para cada tipo de shift
    """
    results = {}
    
    for transform_name, transform_fn in ood_transforms.items():
        correct = 0
        total = 0
        
        for img, label in dataset:
            # Aplicar transform OOD
            img_ood = transform_fn(img)
            
            # Predicción
            with torch.no_grad():
                pred = model(img_ood.unsqueeze(0).to(device)).argmax().item()
            
            correct += (pred == label)
            total += 1
            
            if total >= 100:  # Evaluar en subset
                break
        
        results[transform_name] = correct / total
    
    return results


print("✅ Métrica de Robustez: Accuracy bajo diferentes shifts")
print("\n📊 Ejemplo de reporte:")
print("   Original:  95.2%")
print("   Rotated:   78.5%  ← Gap = 16.7%")
print("   Noisy:     82.1%  ← Gap = 13.1%")
print("   Blurred:   85.3%  ← Gap = 9.9%")
print("\n   Modelo robusto = gaps pequeños!")

---

## 🎓 Resumen y Mejores Prácticas

### ✅ Lo que aprendiste:

1. **OOD** es el gap entre training y deployment
2. Hay diferentes tipos: covariate shift, label shift, concept drift
3. **Meta-Learning** ayuda entrenando en diversidad de distribuciones
4. Combinar: augmentation + domain randomization + meta-learning

### 🛡️ Mejores Prácticas:

#### Durante Training:
- ✅ **Augmentation agresiva**: Simula variabilidad real
- ✅ **Domain randomization**: Entrena en muchas distribuciones
- ✅ **Episodic training**: Support normal, query shifted
- ✅ **Regularización**: Dropout, weight decay, etc.

#### Durante Test:
- ✅ **Test Time Adaptation**: Ajustar con batch normalization adaptativa
- ✅ **Ensemble**: Combinar predicciones de múltiples modelos
- ✅ **Confidence thresholding**: Rechazar predicciones inciertas

#### Monitoreo:
- 📊 **Distribution drift detection**: Monitorear estadísticas de inputs
- 📊 **Performance tracking**: Alertar cuando accuracy cae
- 📊 **Outlier detection**: Identificar ejemplos muy OOD

### 🌟 Aplicaciones Críticas:

**Robótica (Sim-to-Real)**:
- Entrenamiento en simulación (barato, seguro)
- Deployment en mundo real (caro, peligroso)
- Domain randomization crucial!

**Medicina**:
- Entrenar: Hospital A (equipo X, demografía Y)
- Deployer: Hospital B (equipo diferente, demografía diferente)
- Robustez OOD salva vidas!

**Finanzas**:
- Entrenar: Datos históricos (mercado normal)
- Deploy: Crisis, volatilidad extrema
- Fallar = pérdidas masivas

### 📚 Papers Clave:

- **Domain Randomization**: [Domain Randomization for Transferring Deep Neural Networks](https://arxiv.org/abs/1703.06907)
- **Test Time Training**: [Test-Time Training with Self-Supervision](https://arxiv.org/abs/1909.13231)
- **Meta-Learning for OOD**: [Learning to Learn from Failures](https://arxiv.org/abs/2001.10663)

### 🔮 El Futuro:

Meta-Learning + OOD Robustness = **Deployment seguro en el mundo real**

---

## 🎉 ¡Has completado el tutorial de Generalización OOD!

Ahora entiendes uno de los mayores desafíos para IA práctica: funcionar cuando el mundo cambia.
